# CLRR follow-up experiments on Colab A100

Run cells in order. The six main results remain unchanged. This notebook checks their private best-model archives, analyzes existing models, trains four new runs, and writes results to `FiveC/amis-rewire-checkpoints`. Training code and analysis logic live in Python files in the repository.

Add `HF_TOKEN` to Colab Secrets with read/write access to the private model repository before starting.

In [ ]:
# 1. Clone the current code.
import os
import subprocess
from pathlib import Path

REPO_DIR = Path('/content/CLRR')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/HeyDunaX/CLRR.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())

In [ ]:
# 2. Require an A100-class CUDA runtime and load the private-repo token.
import torch
from google.colab import userdata

assert torch.cuda.is_available(), 'Select a GPU runtime (A100 recommended).'
print('GPU:', torch.cuda.get_device_name(0))
assert torch.cuda.get_device_capability(0)[0] >= 8, 'BF16 needs an Ampere-or-newer GPU.'
try:
    secret_token = userdata.get('HF_TOKEN')
except Exception:
    secret_token = None
os.environ['HF_TOKEN'] = secret_token or os.environ.get('HF_TOKEN', '')
assert os.environ['HF_TOKEN'], 'Add HF_TOKEN to Colab Secrets.'
HF_BACKUP_REPO = 'FiveC/amis-rewire-checkpoints'
print('Private backup repo:', HF_BACKUP_REPO)

In [ ]:
# 3. Install into this notebook's Python while keeping Colab's CUDA PyTorch.
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-colab.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'], check=True)
subprocess.run([sys.executable, '-c', 'import amis_rewire; print(amis_rewire.__file__)'], check=True)

In [ ]:
# 4. Verify the exact original data splits.
import csv
for split, expected in [('train', 4600), ('validation', 576), ('test', 575)]:
    path = Path('data/processed') / f'{split}.csv'
    assert path.exists(), f'Missing {path}'
    with path.open(encoding='utf-8', newline='') as stream:
        count = sum(1 for _ in csv.DictReader(stream))
    assert count == expected, f'{split}: expected {expected}, found {count}'
    print(split, count)

## Stage 1 — Check old archives and analyze existing models
The preflight compares saved training arguments to the main-run script before any new training. Missing archives or mismatched settings stop the workflow. The smoke test generates two sentences.

In [ ]:
# 5. Check the five reference archives, then generate two smoke-test translations.
import os
import sys
repo_src = str((Path.cwd() / 'src').resolve())
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)
os.environ['PYTHONPATH'] = repo_src + os.pathsep + os.environ.get('PYTHONPATH', '')
import amis_rewire
print('Using package:', amis_rewire.__file__)
from scripts.followup_analysis import smoke
smoke(HF_BACKUP_REPO, Path('outputs_extra'), Path('data/processed'))

In [ ]:
# 6. Regenerate old-model test predictions and measure encoder cosine.
import importlib
subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
import scripts.followup_analysis as followup_analysis
importlib.reload(followup_analysis)
followup_analysis.analyze(HF_BACKUP_REPO, Path('outputs_extra'), Path('data/processed'), batch_size=16)

## Stage 2 — ByT5-small
Two new runs: Baseline and JEPA+CLRR-Enc. Each run uploads resumable checkpoints, a best-model ZIP, metrics, and test predictions. Re-running this cell skips completed remote runs.

In [ ]:
# 7. Train the ByT5 comparison.
subprocess.run([sys.executable, 'scripts/run_followup_models.py', '--stage', 'byt5', '--repo', HF_BACKUP_REPO], check=True)

## Stage 3 — mT5 rewiring position
Keep JEPA and the main mT5 protocol fixed. Train decoder and both-stack rewiring; compare them to the existing encoder run.

In [ ]:
# 8. Train the two mT5 ablations.
subprocess.run([sys.executable, 'scripts/run_followup_models.py', '--stage', 'ablation', '--repo', HF_BACKUP_REPO], check=True)

## Stage 4 — Export results for the paper
Bootstrap comparisons resample test sentences. They do not measure variation across model-training seeds. Example sentences are candidates for human linguistic review.

In [ ]:
# 9. Create tables, paired bootstrap results, and candidate examples.
subprocess.run([sys.executable, 'scripts/followup_analysis.py', 'report', '--repo', HF_BACKUP_REPO], check=True)
import pandas as pd
from IPython.display import Image, display
analysis_dir = Path('outputs_extra/analysis')
display(pd.read_csv(analysis_dir / 'all_scores.csv'))
display(pd.read_csv(analysis_dir / 'paired_bootstrap.csv'))
display(pd.read_csv(analysis_dir / 'case_candidates.csv').head(10))
display(Image(filename=str(analysis_dir / 'encoder_cosine.png')))
print('Artifacts uploaded to', HF_BACKUP_REPO, 'under analysis/ and checkpoints/<run-name>/')